In [ ]:
# -*- coding: utf-8 -*-
"""scFoundation_finetune_real_data.ipynb

# scFoundation Pretrained Fine-Tuning Benchmark — Phase 8

**What this notebook tests:**
Fine-tuning the *pretrained* scFoundation transformer (100M param xTrimoGene backbone)
on real datasets (80/20 train/test split). Instead of simulated scenarios with replicates,
this notebook iterates over all independent `.h5ad` samples in the `h5ad_real` directory.

**GPU:** L4 (22.5 GB VRAM).
**Crash safety:** Each run saves immediately to Drive. Re-running skips completed runs.
"""

# @title Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scanpy anndata scikit-learn

In [ ]:
# @title Imports & Environment Config
import os
import glob

# ── Allocator config — reduces fragmentation, set before any CUDA ops ─────
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import sys, time, warnings, random, gc, json, math, urllib.request
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.sparse import issparse

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split
from sklearn.metrics import (cohen_kappa_score, f1_score,
                              matthews_corrcoef, accuracy_score)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")

In [ ]:
# @title Configuration & Hyperparameters

# ── EDIT THESE TO MATCH YOUR DRIVE ────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/Thesis FM Computing"
H5AD_DIR        = f"{DRIVE_ROOT}/h5ad_real"
RESULTS_DIR     = f"{DRIVE_ROOT}/phase8_results"
CKPT_DIR        = f"{DRIVE_ROOT}/phase8_checkpoints/scfoundation_finetune"

# Path to the scFoundation pretrained weights (it will download here if missing)
PRETRAINED_CKPT = f"{DRIVE_ROOT}/weights/scFoundation_models.ckpt"

# Path to the ~19k gene list used during scFoundation pretraining (one gene per line)
VOCAB_PATH      = f"{DRIVE_ROOT}/weights/OS_scRNA_gene_index.19264.tsv"
# ─────────────────────────────────────────────────────────────────────────

MODEL_NAME = "scFoundation_pretrained"
TRAIN_SEED = 42

MAX_SEQ_LEN  = 2048

# Lowered learning rate for safe pretrained fine-tuning (prevents NaN divergence)
BATCH_SIZE   = 2
ACCUM_STEPS  = 4
MAX_EPOCHS   = 30
LR           = 2e-5
PATIENCE     = 5
WEIGHT_DECAY = 1e-4

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)

# Dynamically find all samples in the directory
sample_paths = sorted(glob.glob(os.path.join(H5AD_DIR, "*.h5ad")))
sample_paths =[p for p in sample_paths if "_scfoundation" not in p]
SAMPLES =[os.path.basename(p).replace(".h5ad", "") for p in sample_paths]

print(f"Found {len(SAMPLES)} samples in {H5AD_DIR}")
print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"CKPT_DIR    : {CKPT_DIR}")

In [ ]:
# @title scFoundation OFFICIAL model code + weights + 19,264-gene order
#
# FIDELITY FIX (Phase 8): the previous notebook loaded the official checkpoint into a hand-built
# nn.TransformerEncoder whose parameter names do NOT match xTrimoGene -> "0 weights matched" -> the
# benchmark was a RANDOMLY-INITIALISED model. We now use the OFFICIAL model code from the paper repo.
#
# Official repo (Hao et al., Nature Methods 2024): https://github.com/biomap-research/scFoundation
# Official checkpoint (models.ckpt) is requested/downloaded from the SharePoint link in:
#   https://github.com/biomap-research/scFoundation/blob/main/model/README.md
# The genbio-ai HF mirror used below is auto-downloadable in Colab.
# ### VERIFY ON COLAB: confirm the genbio mirror checkpoint is the official xTrimoGene format that
#     load_model_frommmf expects (it reads the model config embedded in the ckpt).
import sys

SCF_REPO_DIR = "/content/scFoundation"
if not os.path.exists(SCF_REPO_DIR):
    print("Cloning official biomap-research/scFoundation ...")
    os.system(f"git clone -q https://github.com/biomap-research/scFoundation {SCF_REPO_DIR}")
# The official xTrimoGene code needs these extra deps (performer.py imports local_attention; einops is
# used across the modules). Installed here so the `from load import ...` below resolves.
os.system("pip install -q local_attention einops")
# Official model loader + helpers live in model/ (load.py: load_model_frommmf, gatherData).
if os.path.join(SCF_REPO_DIR, "model") not in sys.path:
    sys.path.append(os.path.join(SCF_REPO_DIR, "model"))


def download_scfoundation_pretrained():
    """Fetch the pretrained checkpoint. Official source is the SharePoint link in model/README.md;
    the genbio-ai HF mirror is used here only because it is directly downloadable in Colab."""
    MIRROR_URL = "https://huggingface.co/genbio-ai/scFoundation/resolve/main/models.ckpt"
    if not os.path.exists(PRETRAINED_CKPT):
        os.makedirs(os.path.dirname(PRETRAINED_CKPT), exist_ok=True)
        print(f"Checkpoint not found at {PRETRAINED_CKPT}. Downloading mirror (~1.43 GB)...")
        print("  [VERIFY] Confirm this mirror equals the official biomap-research checkpoint.")
        req = urllib.request.Request(MIRROR_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as response, open(PRETRAINED_CKPT, "wb") as out_file:
            while True:
                chunk = response.read(8192 * 1024)
                if not chunk:
                    break
                out_file.write(chunk)
        print("  Successfully downloaded pretrained weights.")
    else:
        print("Pretrained weights found locally.")


def get_scfoundation_genes(vocab_path=VOCAB_PATH, n_genes=19264):
    """Load the official 19,264-gene order (OS_scRNA_gene_index.19264.tsv). The ORDER is load-bearing:
    input-vector position i must be the gene the model was pretrained on. No HGNC fallback."""
    if not os.path.exists(vocab_path):
        raise FileNotFoundError(
            f"scFoundation gene index not found at {vocab_path}. Provide the official "
            f"OS_scRNA_gene_index.19264.tsv (official gene order)."
        )
    df_vocab = pd.read_csv(vocab_path, sep="\t", header=None)
    genes = df_vocab[0].astype(str).tolist()
    if len(genes) < n_genes:
        raise ValueError(f"Expected >= {n_genes} genes in {vocab_path}, got {len(genes)}.")
    genes = genes[:n_genes]
    print(f"Loaded official scFoundation gene order: {len(genes)} genes.")
    return genes


download_scfoundation_pretrained()
SCF_GENES = get_scfoundation_genes()
# gene -> column index in the 19,264 official order. Named SCFOUNDATION_VOCAB so the main loop
# (which passes it positionally and calls len()/.get()) is unchanged.
SCFOUNDATION_VOCAB = {g: i for i, g in enumerate(SCF_GENES)}

In [ ]:
# @title Data Loading & OFFICIAL scFoundation tokenisation
#
# xTrimoGene does NOT take ranked gene-id sequences. It takes a dense 19,264-gene expression vector
# (aligned to the official gene order, normalised + log1p) with TWO read-depth tokens (S, T) appended,
# i.e. a (N, 19266) float matrix. We pack that matrix into the `values` tensor so the existing
# training-loop signature model(ids, values, mask) and the main loop stay unchanged; `ids`/`mask`
# are unused placeholders.
#
# Recipe mirrors model/get_embedding.py (singlecell, --pre_normalized F, --tgthighres a5):
#   per cell:  norm = log1p(counts / counts.sum() * 1e4)
#              S = log10(counts.sum());  T = S + 5      ('a5' = additive high-resolution target)
#              input = concat([norm_aligned_19264, T, S])
# ### VERIFY ON COLAB against model/get_embedding.py: the S/T order, tgthighres value, and whether
#     pre_normalized handling matches your data (these h5ads supply RAW counts via layers['counts']).

def ensure_raw_counts(adata):
    # Raw integer counts are stored in adata.layers['counts']; adata.X is z-scored (has negatives).
    # Prefer the counts layer, then .raw, then fall back to X (guarded). This replaces the old
    # destructive np.clip(X, 0, None) workaround, which zeroed all below-mean z-scored values.
    if "counts" in adata.layers:
        X, var = adata.layers["counts"], adata.var
    elif adata.raw is not None:
        X, var = adata.raw.X, adata.raw.var
    else:
        X, var = adata.X, adata.var
    if issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float32)
    if X.min() < 0:
        raise ValueError(
            "Count matrix has negative values \u2014 layers['counts'] is not raw counts "
            "(adata.X is z-scored). Cannot tokenize."
        )
    X = np.round(X).astype(np.float32)
    return ad.AnnData(
        X   = X,
        obs = adata.obs.copy(),
        var = var.copy(),
        uns = adata.uns.copy()
    )

def load_sample(sample_name):
    raw_path = os.path.join(H5AD_DIR, f"{sample_name}.h5ad")
    if not os.path.exists(raw_path):
        raise FileNotFoundError(f"Cannot find: {raw_path}")

    adata = sc.read_h5ad(raw_path)
    adata = ensure_raw_counts(adata)

    if "cell_type" not in adata.obs.columns:
        if "Ground_Truth_Celltype" in adata.obs.columns:
            adata.obs["cell_type"] = adata.obs["Ground_Truth_Celltype"]
        else:
            assert False, (f"'cell_type' column missing. Available: {list(adata.obs.columns)}")

    n_types = adata.obs["cell_type"].nunique()
    print(f"  Loaded {sample_name}: {adata.shape[0]} cells \u00d7 {adata.shape[1]} genes | {n_types} cell types")
    return adata

def stratified_split(adata, test_size=0.2):
    initial_count = len(adata)
    adata = adata[~adata.obs["cell_type"].isna()].copy()
    if len(adata) < initial_count:
        print(f"  [FIX] Removed {initial_count - len(adata)} cells with missing cell types.")

    counts = adata.obs["cell_type"].value_counts()
    singletons = counts[counts < 2].index.tolist()
    if singletons:
        adata = adata[~adata.obs["cell_type"].isin(singletons)].copy()

    labels = adata.obs["cell_type"].values
    idx    = np.arange(len(labels))
    tr_idx, te_idx = train_test_split(idx, test_size=test_size, stratify=labels, random_state=TRAIN_SEED)
    return adata[tr_idx].copy(), adata[te_idx].copy()

def encode_labels(train_a, test_a):
    all_types = sorted(train_a.obs["cell_type"].unique())
    label2id  = {t: i for i, t in enumerate(all_types)}
    id2label  = {i: t for t, i in label2id.items()}
    y_train   = np.array([label2id[t] for t in train_a.obs["cell_type"]])
    y_test    = np.array([label2id[t] for t in test_a.obs["cell_type"]])
    return label2id, id2label, y_train, y_test

def to_dense(adata):
    X = adata.X
    if issparse(X): X = X.toarray()
    return X.astype(np.float32)

def tokenise_for_scfoundation(X_raw, gene_names, gene_to_idx, n_genes=19264):
    n_cells = X_raw.shape[0]
    matched = sum(1 for g in gene_names if g in gene_to_idx)
    print(f"  Matched to scFoundation 19264-gene order: {matched}/{len(gene_names)} "
          f"({100.0*matched/len(gene_names):.1f}%)")
    if matched == 0:
        raise ValueError("No genes matched the scFoundation 19264-gene order for this sample.")

    # Align raw counts into the official gene order.
    aligned  = np.zeros((n_cells, n_genes), dtype=np.float32)
    cols_src = [i for i, g in enumerate(gene_names) if g in gene_to_idx]
    cols_dst = [gene_to_idx[gene_names[i]] for i in cols_src]
    aligned[:, cols_dst] = X_raw[:, cols_src]

    total = aligned.sum(axis=1, keepdims=True)                  # per-cell read depth (raw counts)
    total = np.where(total == 0, 1.0, total)
    norm  = np.log1p(aligned / total * 1e4).astype(np.float32)  # normalise + log1p
    S = np.log10(total).astype(np.float32)                      # read-depth token S
    T = (S + 5.0).astype(np.float32)                            # tgthighres 'a5' (additive)
    x = np.concatenate([norm, T, S], axis=1).astype(np.float32) # (N, 19266)

    values = torch.tensor(x, dtype=torch.float32)
    ids    = torch.zeros((n_cells, 1), dtype=torch.long)   # placeholder (loop signature compat)
    mask   = torch.zeros((n_cells, 1), dtype=torch.bool)   # placeholder
    return ids, values, mask

In [ ]:
# @title scFoundation model: OFFICIAL xTrimoGene encoder + classification head
#
# FIDELITY FIX (Phase 8): replaces the hand-built nn.TransformerEncoder (which loaded 0 official
# weights -> random init) with the OFFICIAL model built by model/load.py::load_model_frommmf from the
# cloned repo. xTrimoGene is an asymmetric encoder-decoder; downstream tasks use the ENCODER (per the
# paper), so we pool the encoder output into a cell embedding and attach a classification head.
#
# ### VERIFY ON COLAB: the loader call and the encoder-forward/pooling block below are transcribed
#     from model/get_embedding.py. On your first run, reconcile against the cloned repo:
#       - load_model_frommmf signature + the `key` value ("cell" vs "gene"),
#       - the embedded-config key paths (pad_token_id, hidden dim),
#       - token_emb / pos_emb call signatures and gatherData usage,
#       - the pooling (--pool_type) used to form the per-cell embedding.
from load import load_model_frommmf, gatherData   # official helpers (biomap-research/scFoundation/model)


class scFoundationPretrained(nn.Module):
    def __init__(self, vocab_size, num_classes, pad_token_id, pretrained_path=None):
        super().__init__()
        if not (pretrained_path and os.path.exists(pretrained_path)):
            raise FileNotFoundError(f"scFoundation checkpoint not found: {pretrained_path}")

        # Build the official encoder from the checkpoint (config is embedded in the ckpt).
        # ### VERIFY: `key` selects the sub-model. "cell" is the canonical choice for cell-level
        #     embeddings in get_embedding.py; if loading fails try "gene", "all", or None (varies by
        #     repo revision). This is the most likely first-run failure point.
        self.backbone, self.model_config = load_model_frommmf(pretrained_path, key="cell")
        self.pad_id  = self.model_config.get("pad_token_id", 19264)            # ### VERIFY key
        self.d_model = self.model_config["encoder"]["hidden_dim"]              # ### VERIFY key path

        # Count how much of the official model actually loaded (this used to be 0).
        n_backbone = sum(p.numel() for p in self.backbone.parameters())
        print(f"  scFoundation OFFICIAL xTrimoGene encoder loaded: {n_backbone:,} backbone params.")
        if n_backbone == 0:
            raise RuntimeError("Official scFoundation encoder has 0 params \u2014 checkpoint/loader mismatch.")

        self.cls_head = nn.Sequential(
            nn.Linear(self.d_model, self.d_model),
            nn.GELU(),
            nn.LayerNorm(self.d_model),
            nn.Linear(self.d_model, num_classes),
        )
        n = sum(p.numel() for p in self.parameters())
        print(f"  scFoundation classifier: {num_classes} classes | total {n:,} params")

    def _encode(self, x):
        # x: (B, 19266) float = 19264 aligned genes + T + S. Mirrors model/get_embedding.py.
        value_labels = x > 0
        x_gathered, x_padding = gatherData(x, value_labels, self.pad_id)
        device   = x.device
        gene_ids = torch.arange(x.shape[1], device=device).repeat(x.shape[0], 1)
        pos_ids, _ = gatherData(gene_ids, value_labels, self.pad_id)

        x_emb = self.backbone.token_emb(torch.unsqueeze(x_gathered, 2).float(), output_weight=0)
        x_emb = x_emb + self.backbone.pos_emb(pos_ids)
        geneemb = self.backbone.encoder(x_emb, x_padding)        # (B, L, d_model)

        # Cell embedding: last two positions hold the T/S read-depth tokens after attention; the
        # official get_embedding.py concatenates/pools these. We mean-pool all token embeddings.
        cell_emb = geneemb.mean(dim=1)                            # ### VERIFY pooling vs --pool_type
        return cell_emb

    def forward(self, input_ids, values, src_key_padding_mask=None):
        # `values` carries the official (B, 19266) input; ids/mask are unused placeholders.
        cell_emb = self._encode(values)
        return self.cls_head(cell_emb)

In [ ]:
# @title Training, Evaluation, and Save Utilities

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total, n = 0.0, 0
    optimizer.zero_grad()

    # Use bfloat16 if supported by GPU (like L4) as it prevents gradient overflow perfectly
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    for step, (ids, vals, mask, yb) in enumerate(loader):
        ids, vals, mask, yb = ids.to(DEVICE), vals.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda"), dtype=amp_dtype):
            logits = model(ids, vals, src_key_padding_mask=mask)
            loss = criterion(logits.float(), yb) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total += loss.item() * ACCUM_STEPS * len(yb)
        n += len(yb)

    return total / n

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total, n, preds = 0.0, 0,[]
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    for ids, vals, mask, yb in loader:
        ids, vals, mask, yb = ids.to(DEVICE), vals.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda"), dtype=amp_dtype):
            logits = model(ids, vals, src_key_padding_mask=mask)

        total += criterion(logits.float(), yb).item() * len(yb); n += len(yb)
        preds.extend(logits.argmax(1).cpu().numpy())
    return total / n, np.array(preds)

def fit_model(model, X_tr_ids, X_tr_vals, X_tr_mask, y_tr, X_te_ids, X_te_vals, X_te_mask, y_te, sample_name, n_classes):
    tr_dl = DataLoader(TensorDataset(X_tr_ids, X_tr_vals, X_tr_mask, torch.tensor(y_tr, dtype=torch.long)),
                       batch_size=BATCH_SIZE, shuffle=True, pin_memory=(DEVICE.type == "cuda"))
    te_dl = DataLoader(TensorDataset(X_te_ids, X_te_vals, X_te_mask, torch.tensor(y_te, dtype=torch.long)),
                       batch_size=BATCH_SIZE, shuffle=False, pin_memory=(DEVICE.type == "cuda"))

    # CRITICAL FIX: Safe, bounded class weights to prevent 1,000,000x scaling causing NaNs
    counts = np.bincount(y_tr, minlength=n_classes)
    w = len(y_tr) / (n_classes * np.maximum(counts, 1))
    w = np.clip(w, 0.1, 10.0)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(DEVICE))

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    ckpt = os.path.join(CKPT_DIR, f"scfoundation_finetune_{sample_name}_best.pt")

    best_val, pat = float("inf"), 0
    ckpt_saved = False
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss = train_epoch(model, tr_dl, optimizer, criterion)
        vl_loss, _ = eval_epoch(model, te_dl, criterion)
        scheduler.step()

        if not math.isnan(vl_loss) and vl_loss < best_val:
            best_val, pat = vl_loss, 0
            torch.save(model.state_dict(), ckpt)
            ckpt_saved = True
        else:
            pat += 1

        if epoch % 5 == 0:
            print(f"    Epoch {epoch:02d} | train={tr_loss:.4f} val={vl_loss:.4f} | patience={pat}/{PATIENCE}")

        if pat >= PATIENCE:
            print(f"    Early stop at epoch {epoch}")
            break

    if ckpt_saved:
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    else:
        print(f"    [WARN] No valid checkpoint saved. Using model state from last epoch.")

    _, final_preds = eval_epoch(model, te_dl, criterion)

    runtime = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE.type == "cuda" else 0
    print(f"    Done in {runtime/60:.1f} min | peak GPU: {peak_mb:.0f} MB")
    return final_preds, runtime, peak_mb

In [ ]:
def compute_metrics(y_true, y_pred, id2label):
    kappa    = cohen_kappa_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc      = matthews_corrcoef(y_true, y_pred)
    acc      = accuracy_score(y_true, y_pred)

    unique, counts = np.unique(y_true, return_counts=True)
    rarest = unique[np.argmin(counts)]
    rare_f1 = f1_score((y_true == rarest).astype(int),
                       (y_pred == rarest).astype(int), zero_division=0)
    return {
        "kappa":          round(float(kappa),    4),
        "macro_f1":       round(float(macro_f1), 4),
        "mcc":            round(float(mcc),      4),
        "accuracy":       round(float(acc),      4),
        "rare_type_f1":   round(float(rare_f1),  4),
        "rare_type_name": id2label[rarest],
        "rare_type_n":    int(counts.min()),
        "n_test":         int(len(y_true)),
        "n_classes":      int(len(id2label)),
    }

def print_metrics(m):
    print(f"  κ (kappa)    : {m['kappa']:.4f}")
    print(f"  Macro F1     : {m['macro_f1']:.4f}")
    print(f"  MCC          : {m['mcc']:.4f}")
    print(f"  Accuracy     : {m['accuracy']:.4f}")
    print(f"  Rare type F1 : {m['rare_type_f1']:.4f}  ({m['rare_type_name']}, n_test={m['rare_type_n']})")

In [ ]:
CSV_PATH = os.path.join(RESULTS_DIR, "phase8_results_scFoundation.csv")

def save_result(sample_name, metrics, predictions, runtime_s, peak_gpu_mb):
    row = {
        "model":       MODEL_NAME,
        "sample":      sample_name,
        "condition":   "pretrained_finetune",
        "runtime_s":   round(runtime_s, 1),
        "peak_gpu_mb": round(peak_gpu_mb, 0),
        **metrics
    }
    row_df = pd.DataFrame([row])
    if os.path.exists(CSV_PATH):
        row_df.to_csv(CSV_PATH, mode="a", header=False, index=False)
    else:
        row_df.to_csv(CSV_PATH, mode="w", header=True,  index=False)

    pred_fname = f"{MODEL_NAME}_{sample_name}_preds.npy"
    np.save(os.path.join(RESULTS_DIR, pred_fname), predictions)
    print(f"  Saved metrics → phase8_results_scFoundation.csv")
    print(f"  Saved preds   → {pred_fname}")

def check_done(sample_name):
    if not os.path.exists(CSV_PATH): return False
    df = pd.read_csv(CSV_PATH)
    return ((df["model"]  == MODEL_NAME)  &
            (df["sample"] == sample_name) &
            (df["condition"] == "pretrained_finetune")).any()

In [ ]:
# @title Main Execution Loop

completed, total = 0, len(SAMPLES)

print("=" * 65)
print(f"scFoundation Pretrained Fine-Tuning | {total} real samples")
print("=" * 65)

for sample in SAMPLES:
    print(f"\n{'─'*65}")
    print(f"[{completed+1}/{total}]  Sample: {sample}")
    print(f"{'─'*65}")

    if DEVICE.type == "cuda":
        free_gb, total_gb =[x/1e9 for x in torch.cuda.mem_get_info()]
        print(f"  GPU memory: {free_gb:.1f} GB free / {total_gb:.1f} GB total")

    if check_done(sample):
        print(f"[SKIP] Already in results CSV")
        completed += 1
        continue

    try:
        adata = load_sample(sample)
    except FileNotFoundError as e:
        print(f"  [WARN] {e}\n  Skipping sample.")
        continue

    train_a, test_a = stratified_split(adata)
    label2id, id2label, y_train, y_test = encode_labels(train_a, test_a)
    n_classes = len(label2id)
    print(f"  Train: {len(y_train)} cells | Test: {len(y_test)} cells | Classes: {n_classes}")

    print("  Tokenising...")
    gene_names = list(adata.var_names)

    X_train_dense = to_dense(train_a)
    del train_a; gc.collect()
    X_tr_ids, X_tr_vals, X_tr_mask = tokenise_for_scfoundation(X_train_dense, gene_names, SCFOUNDATION_VOCAB)
    del X_train_dense; gc.collect()

    X_test_dense = to_dense(test_a)
    del adata, test_a; gc.collect()
    X_te_ids, X_te_vals, X_te_mask = tokenise_for_scfoundation(X_test_dense, gene_names, SCFOUNDATION_VOCAB)
    del X_test_dense; gc.collect()

    VOCAB_SIZE = len(SCFOUNDATION_VOCAB)
    PAD_TOKEN_ID = SCFOUNDATION_VOCAB.get("<pad>", 0)

    model = scFoundationPretrained(
        vocab_size=VOCAB_SIZE,
        num_classes=n_classes,
        pad_token_id=PAD_TOKEN_ID,
        pretrained_path=PRETRAINED_CKPT
    ).to(DEVICE)

    print("  Training...")
    preds, runtime, peak_mb = fit_model(
        model, X_tr_ids, X_tr_vals, X_tr_mask, y_train,
        X_te_ids, X_te_vals, X_te_mask, y_test, sample, n_classes
    )

    metrics = compute_metrics(y_test, preds, id2label)
    print("  Results:")
    print_metrics(metrics)

    save_result(sample, metrics, preds, runtime, peak_mb)
    completed += 1

    del model, X_tr_ids, X_tr_vals, X_tr_mask, X_te_ids, X_te_vals, X_te_mask
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "=" * 65)
print(f"Complete: {completed}/{total} runs")
print("=" * 65)

In [ ]:
# @title Print Results Summary

if os.path.exists(CSV_PATH):
    df   = pd.read_csv(CSV_PATH)
    mine = df[df["model"] == MODEL_NAME].copy()

    if len(mine) > 0:
        print("=== Per-run results ===")
        cols =["sample", "kappa", "macro_f1", "rare_type_f1", "accuracy", "runtime_s", "peak_gpu_mb"]
        print(mine[cols].sort_values("sample").to_string(index=False))

        print(f"\nTotal samples recorded: {len(mine)}/{total}")
        print(f"Total runtime         : {mine['runtime_s'].sum()/3600:.2f} hours")
        print(f"Mean κ (all samples)  : {mine['kappa'].mean():.4f}")
    else:
        print("No results yet — run the main loop first.")
else:
    print(f"No CSV found at {CSV_PATH}")

In [ ]:
from google.colab import runtime
runtime.unassign()